In [3]:
# Install necessary packages
!pip install requests==2.31.0
!pip install google-cloud-aiplatform[adk] --upgrade

  Using cached requests-2.31.0-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.31.0-py3-none-any.whl (62 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.31.0 which is incompatible.
datasets 4.0.0 requires requests>=2.32.2, but you have requests 2.31.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.31.0 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.2

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
^C


In [ ]:
import requests
import json
from google.adk.agents import Agent
from vertexai import agent_engines

# --- API Key Placeholders ---
GOOGLE_MAPS_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
NWS_USER_AGENT = "myweatherapp.com, contact@myweatherapp.com"

In [16]:
def get_weather_forecast(location_name: str) -> str:
    """
    Fetches the weather forecast for a given location.

    Args:
        location_name: The name of the location (e.g., "Seattle", "London").

    Returns:
        A string containing the weather forecast or an error message.
    """
    # First, get coordinates for the location
    coords = get_coordinates_google_maps(location_name)
    if not coords:
        return f"Sorry, I couldn't find a location called '{location_name}'."

    latitude, longitude = coords

    # Then, get the NWS forecast using the coordinates
    return fetch_nws_forecast(latitude, longitude)

def get_coordinates_google_maps(location_name: str) -> tuple[float, float] | None:
    """
    Helper function to convert a location name into latitude and longitude.
    This function is NOT a tool itself but is used by the get_weather_forecast tool.
    """
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location_name, "key": GOOGLE_MAPS_API_KEY}
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        if data["status"] == "OK" and data["results"]:
            location = data["results"][0]["geometry"]["location"]
            return location["lat"], location["lng"]
    except (requests.exceptions.RequestException, json.JSONDecodeError, KeyError):
        return None
    return None

def fetch_nws_forecast(latitude: float, longitude: float) -> str:
    """
    Helper function to fetch the forecast from the NWS.
    This function is NOT a tool itself but is used by the get_weather_forecast tool.
    """
    headers = {"User-Agent": NWS_USER_AGENT, "Accept": "application/geo+json"}
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        response = requests.get(points_url, headers=headers)
        response.raise_for_status()
        point_data = response.json()
        forecast_url = point_data.get("properties", {}).get("forecast")

        if not forecast_url:
            return "Could not find forecast URL."

        response = requests.get(forecast_url, headers={"User-Agent": NWS_USER_AGENT, "Accept": "application/json"})
        response.raise_for_status()
        forecast_data = response.json()
        periods = forecast_data.get("properties", {}).get("periods", [])

        if not periods:
            return "No forecast periods available."

        summary = [f"**{p.get('name')}**: {p.get('shortForecast')}. Temp: {p.get('temperature')}°{p.get('temperatureUnit')}." for p in periods[:3]]
        return "\n".join(summary)
    except (requests.exceptions.RequestException, json.JSONDecodeError, KeyError) as e:
        return f"Error fetching NWS data: {e}"

In [17]:
# Instantiate the agent
weather_agent = Agent(
    model="gemini-1.5-flash",
    name='weather_forecaster',
    tools=[get_weather_forecast],
)

# Create an AdkApp instance for local testing
app = agent_engines.AdkApp(agent=weather_agent)

In [18]:
# --- Interactive Chat Loop ---
def run_chat(app):
    print("Weather Agent is ready. Ask for the weather (e.g., 'what is the weather in Seattle?'). Type 'bye' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "bye":
            print("It was nice chatting with you!")
            break

        final_answer = get_weather_forecast(user_input)
        print(f"Weather Agent: {final_answer}")

# Start the chat
run_chat(app)

Weather Agent is ready. Ask for the weather (e.g., 'what is the weather in Seattle?'). Type 'bye' to exit.
You: Dallas
Weather Agent: **Today**: Partly Sunny. Temp: 79°F.
**Tonight**: Partly Cloudy. Temp: 62°F.
**Tuesday**: Partly Sunny. Temp: 84°F.
You: Tell me the weather in Seattle
Weather Agent: **Today**: Mostly Sunny. Temp: 58°F.
**Tonight**: Mostly Cloudy then Chance Light Rain. Temp: 47°F.
**Tuesday**: Light Rain. Temp: 53°F.
You: bye
It was nice chatting with you!


In [19]:
# Test code to check weather for multiple cities
cities_to_test = ["New York", "Miami", "Los Angeles"]

print("--- Testing Weather Forecast for Multiple Cities ---")
for city in cities_to_test:
    print(f"\nWeather for {city}:")
    forecast = get_weather_forecast(city)
    print(forecast)
print("----------------------------------------------------")

--- Testing Weather Forecast for Multiple Cities ---

Weather for New York:
**Today**: Mostly Sunny. Temp: 31°F.
**Tonight**: Partly Cloudy. Temp: 27°F.
**Tuesday**: Rain And Snow. Temp: 43°F.

Weather for Miami:
**Today**: Chance Showers And Thunderstorms. Temp: 78°F.
**Tonight**: Slight Chance Rain Showers. Temp: 72°F.
**Tuesday**: Slight Chance Rain Showers then Slight Chance Showers And Thunderstorms. Temp: 79°F.

Weather for Los Angeles:
**Today**: Mostly Sunny. Temp: 76°F.
**Tonight**: Partly Cloudy then Patchy Fog. Temp: 50°F.
**Tuesday**: Patchy Fog then Mostly Sunny. Temp: 78°F.
----------------------------------------------------
